# Road Surface Type Classification Task

## Phase 1

### Section 1: List of requirements

This section records the software environment and package versions required to reproduce the dataset audit and Phase 1 exploratory data analysis on the JupyterHub VM.

In [1]:
import sys                                                                                                                              
from pathlib import Path                                                                                                                
import pandas as pd                                                                                                                     
                                                                                                                                        
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()                                                                
if str(ROOT_DIR) not in sys.path:                                                                                                       
    sys.path.insert(0, str(ROOT_DIR))                                                                                                   
                                                                                                                                        
from src.helper.inventory import get_environment_info                                                                                   
                                                                                                                                        
env_info = get_environment_info()                                                                                                       
pd.DataFrame(list(env_info["packages"].items()), columns=["Package", "Version"])

,Package,Version
0,numpy,2.3.5
1,pandas,3.0.6
2,opencv-python,not installed
3,pillow,12.3.0


### Section 2: Description of the dataset

StreetSurfaceVis contains 9,122 crowdsourced street-level images collected through Mapillary from locations within Germany. Each image is manually labeled for surface type and surface quality ([Kapp et al., 2025](https://doi.org/10.1038/s41597-024-04295-9)). 

Our task predicts surface type across five classes: asphalt, concrete, paving stones, sett, and unpaved.

In [2]:
import os                                                                                                                                                
                                                                                                                                                            
BASE_DATA = ROOT_DIR / "data"                                                                                                                            
METADATA_PATH = str(BASE_DATA / "streetSurfaceVis_v1_0.csv")                                                                                             
DATA_PATH = str(BASE_DATA / "s_256") 

### Section 3: Exploratory data analysis

We audit the StreetSurfaceVis v1.0 release metadata table. We verify required fields, non-null values, unique image identifiers, valid class labels, and the published train/test assignment partition counts.

In [3]:
from src.helper.inventory import load_and_validate_metadata
  
metadata_df, metadata_summary = load_and_validate_metadata(METADATA_PATH)
pd.DataFrame([metadata_summary]).T.rename(columns={0: "Value"})

,Value
total_records,9122
expected_total,9122
train_records,8346
expected_train,8346
test_records,776
expected_test,776
unique_ids,9122
duplicate_ids,0
has_nulls,False
unexpected_classes,[]


Next, we audit the s_256 image directory against the CSV metadata. Each image file is validated for presence and integrity without holding entire images in memory, reconciling readable counts and exposing the inventory.

In [4]:
from src.helper.inventory import audit_image_inventory

inventory_df, audit_summary, missing_ids, decode_fails, extra_files = (
    audit_image_inventory(metadata_df, DATA_PATH)
)
pd.DataFrame([audit_summary]).T.rename(columns={0: "Count"})

,Count
expected_images,9122
found_images,9122
readable_images,9120
missing_images,0
decode_failures,2
extra_files,0
unusable_images,2


We inspect the decode failures identified during the image audit, displaying their image identifiers, failure reasons, and corresponding metadata records.

In [5]:
decode_df = pd.DataFrame(
    decode_fails, columns=["mapillary_image_id", "decode_error", "file_size_bytes"]
)
metadata_df[metadata_df["mapillary_image_id"].isin(decode_df["mapillary_image_id"])].merge(
    decode_df, on="mapillary_image_id"
)

,mapillary_image_id,user_id,user_name,captured_at,longitude,latitude,train,surface_type,surface_quality,decode_error,file_size_bytes
0,3865590250223952,103078715264357,hotteliene,1583081300392,10.581377,53.325355,True,concrete,excellent,cannot identify image file '/home/jupyter-csc9...,0
1,2289217127888486,106289114939007,goobersmash,1461932090212,13.035185,52.404632,True,sett,good,cannot identify image file '/home/jupyter-csc9...,0


We check for any missing image files or extra untracked files in the image directory to ensure complete dataset alignment.

In [6]:
file_discrepancies = {
    "missing_image_ids": missing_ids,
    "extra_filenames": extra_files,
}
pd.DataFrame([file_discrepancies]).T.rename(columns={0: "Discrepancies"})

,Discrepancies
missing_image_ids,[]
extra_filenames,[]


The audit reconciles 9,122 expected records with 9,120 readable images, 2 decode failures of size 0 bytes, and 0 missing or extra files. The verified inventory dataframe is preserved for subsequent split and balance checks.

### Section 4: Dataset Split

### Section 5: Traditional ML model training

## 